<a href="https://colab.research.google.com/github/RaghadTwn1/FastAPI-LLM-Comparison/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')

In [ ]:
import os

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello"
)

print(response.text)

In [ ]:
!git clone https://github.com/fastapi/fastapi.git

In [ ]:
!ls fastapi/docs/en/docs

In [ ]:
!find fastapi/docs/en/docs/tutorial -name "*.md" | wc -l
!find fastapi/docs/en/docs/advanced -name "*.md" | wc -l

In [ ]:
with open("fastapi/docs/en/docs/tutorial/first-steps.md", "r", encoding="utf-8") as f:
    content = f.read()

print(content[:500])

In [ ]:
!ls fastapi/docs_src/first_steps

In [ ]:
import re

pattern = r'\{\*\s*(.*?)\s*\*\}'
matches = re.findall(pattern, content)
print(matches)

In [ ]:
clean_paths = [m.split(" ")[0] for m in matches]
print(clean_paths)

In [ ]:
fixed_path = "fastapi/docs_src/first_steps/tutorial001_py310.py"

with open(fixed_path, "r", encoding="utf-8") as f:
    code = f.read()

print(code)

In [ ]:
import os

def replace_code_references(md_content, md_file_path):
    pattern = r'\{\*\s*(.*?)\s*\*\}'
    matches = re.findall(pattern, md_content)

    for m in matches:
        rel_path = m.split(" ")[0]
        real_path = os.path.normpath(os.path.join(os.path.dirname(md_file_path), rel_path))

        try:
            with open(real_path, "r", encoding="utf-8") as f:
                code = f.read()
            code_block = f"```python\n{code}\n```"
            md_content = md_content.replace("{* " + m + " *}", code_block)
        except FileNotFoundError:
            pass

    return md_content

In [ ]:
md_path = "fastapi/docs/en/docs/tutorial/first-steps.md"

with open(md_path, "r", encoding="utf-8") as f:
    original_content = f.read()

final_content = replace_code_references(original_content, md_path)

print(final_content[:1500])

In [ ]:
rel_path = "../../docs_src/first_steps/tutorial001_py310.py"
computed_path = os.path.normpath(os.path.join(os.path.dirname(md_path), rel_path))
print(computed_path)

In [ ]:
def replace_code_references(md_content, md_file_path):
    pattern = r'\{\*\s*(.*?)\s*\*\}'
    matches = re.findall(pattern, md_content)

    for m in matches:
        rel_path = m.split(" ")[0]
        # نستخرج بس الجزء اللي بعد "docs_src/"
        part_after = rel_path.split("docs_src/")[-1]
        real_path = os.path.join("fastapi", "docs_src", part_after)

        try:
            with open(real_path, "r", encoding="utf-8") as f:
                code = f.read()
            code_block = f"```python\n{code}\n```"
            md_content = md_content.replace("{* " + m + " *}", code_block)
        except FileNotFoundError:
            pass

    return md_content

In [ ]:
final_content = replace_code_references(original_content, md_path)
print(final_content[:1500])

In [ ]:
import re

def clean_html(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

In [ ]:
cleaned = clean_html(final_content)
print(cleaned[:1500])

In [ ]:
import glob

tutorial_files = glob.glob("fastapi/docs/en/docs/tutorial/**/*.md", recursive=True)
advanced_files = glob.glob("fastapi/docs/en/docs/advanced/**/*.md", recursive=True)

all_files = tutorial_files + advanced_files

print(len(all_files))
print(all_files[:5])

In [ ]:
dataset = []

for path in all_files:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    with_code = replace_code_references(raw, path)
    clean_text = clean_html(with_code)

    title = clean_text.split("\n")[0].replace("#", "").strip()

    dataset.append({
        "title": title,
        "source_path": path,
        "content": clean_text
    })

print(len(dataset))
print(dataset[0])

In [ ]:
empty_count = 0
short_count = 0

for item in dataset:
    if len(item["content"]) == 0:
        empty_count += 1
    elif len(item["content"]) < 200:
        short_count += 1

print("عدد الملفات الفاضية:", empty_count)
print("عدد الملفات القصيرة جداً (أقل من 200 حرف):", short_count)

In [ ]:
import json

with open("fastapi_dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print("تم الحفظ")

In [ ]:
!ls -lh fastapi_dataset.json